# 06 – Gradient Boosting
**CS 5331/7331 · Project Team Five**

Trains a Gradient Boosting classifier on both feature sets (full 30 / no-external 25), with and without SMOTEN.
Hyperparameters are tuned with 5-fold stratified CV on the training set (scored on F1), then each best model is scored once on the test set.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q imbalanced-learn

import os, json
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTEN

BASE = '/content/drive/MyDrive/CS7331_Phishing'
DATA_DIR = f'{BASE}/data'
RESULTS_DIR = f'{BASE}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
SEED = 42

# load the exact splits Safia saved from preprocessing
X_train = pd.read_csv(f'{DATA_DIR}/X_train.csv')
X_test = pd.read_csv(f'{DATA_DIR}/X_test.csv')
y_train = pd.read_csv(f'{DATA_DIR}/y_train.csv')['phishing']
y_test = pd.read_csv(f'{DATA_DIR}/y_test.csv')['phishing']
with open(f'{DATA_DIR}/feature_sets.json') as f:
    FEATURE_SETS = json.load(f)

print(X_train.shape, X_test.shape, y_train.mean().round(3))

Mounted at /content/drive
(4679, 30) (1170, 30) 0.516


In [2]:
from joblib import Memory
CACHE = Memory('/content/smote_cache', verbose=0)  # remembers SMOTEN results so it doesn't redo the same thing
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Gradient Boosting hyperparams:
#  - n_estimators = how many trees to add in sequence (more = better but slower, risk of overfitting)
#  - max_depth = how deep each individual tree is (shallow trees are the point of boosting — 3-5 is typical)
#  - learning_rate = how much each tree contributes (lower = more trees needed but better generalization)
GRID = {
    'gb__n_estimators': [100, 200],
    'gb__max_depth': [3, 5],
    'gb__learning_rate': [0.05, 0.1],
}

def make_pipe(use_smote):
    steps = []
    if use_smote:
        steps.append(('smote', SMOTEN(random_state=SEED)))  # inside the pipeline so it only touches training folds, test set stays untouched
    steps.append(('gb', GradientBoostingClassifier(random_state=SEED)))
    return Pipeline(steps, memory=CACHE)

In [3]:
rows = []
preds = pd.DataFrame({'y_true': y_test.values})

for fs_name, cols in FEATURE_SETS.items():
    for use_smote in [False, True]:
        search = GridSearchCV(make_pipe(use_smote), GRID, scoring='f1', cv=cv, n_jobs=-1)
        search.fit(X_train[cols], y_train)
        print(f"done: GradientBoosting | {fs_name} | {'smote' if use_smote else 'no_smote'}")

        best = search.best_estimator_  # refit on the full training set with the best setting
        y_pred = best.predict(X_test[cols])
        y_prob = best.predict_proba(X_test[cols])[:, 1]

        smote_tag = 'smote' if use_smote else 'no_smote'
        rows.append({
            'model': 'GradientBoosting', 'features': fs_name, 'smote': smote_tag,
            'best_params': search.best_params_,
            'cv_f1': round(search.best_score_, 4),
            'accuracy': round(accuracy_score(y_test, y_pred), 4),
            'precision': round(precision_score(y_test, y_pred), 4),
            'recall': round(recall_score(y_test, y_pred), 4),
            'f1': round(f1_score(y_test, y_pred), 4),
            'roc_auc': round(roc_auc_score(y_test, y_prob), 4),
        })
        # keep predictions for the confusion matrices and ROC curves later
        key = f'GradientBoosting|{fs_name}|{smote_tag}'
        preds[f'{key}|pred'] = y_pred
        preds[f'{key}|prob'] = y_prob

results = pd.DataFrame(rows)
results.sort_values('cv_f1', ascending=False)

done: GradientBoosting | full | no_smote
done: GradientBoosting | full | smote
done: GradientBoosting | no_external | no_smote
done: GradientBoosting | no_external | smote


,model,features,smote,best_params,cv_f1,accuracy,precision,recall,f1,roc_auc
1,GradientBoosting,full,smote,"{'gb__learning_rate': 0.1, 'gb__max_depth': 5,...",0.9553,0.9496,0.9642,0.9371,0.9505,0.9934
0,GradientBoosting,full,no_smote,"{'gb__learning_rate': 0.1, 'gb__max_depth': 5,...",0.9538,0.9556,0.9678,0.9454,0.9564,0.9939
2,GradientBoosting,no_external,no_smote,"{'gb__learning_rate': 0.1, 'gb__max_depth': 5,...",0.9502,0.9359,0.9506,0.9238,0.9370,0.9886
3,GradientBoosting,no_external,smote,"{'gb__learning_rate': 0.1, 'gb__max_depth': 5,...",0.9500,0.9342,0.9459,0.9255,0.9356,0.9875


In [4]:
results.to_csv(f'{RESULTS_DIR}/gb_results.csv', index=False)
preds.to_csv(f'{RESULTS_DIR}/gb_predictions.csv', index=False)
print("Saved gb_results.csv and gb_predictions.csv to /results")

Saved gb_results.csv and gb_predictions.csv to /results
